In [ ]:
!pip install PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 66.2 MB/s eta 0:00:00


In [ ]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.6 MB/s eta 0:00:00


In [ ]:
import requests
from tqdm import tqdm

url = "https://belnorth.com/wp-content/uploads/2019/04/fifa11manual.pdf"
file_name = "fifa_11_plus_manual.pdf"

response = requests.get(url, stream=True)
total_size = int(response.headers.get('content-length', 0))

with open(file_name, "wb") as file, tqdm(
    desc=file_name,
    total=total_size,
    unit='B',
    unit_scale=True,
    unit_divisor=1024,
) as bar:
    for data in response.iter_content(chunk_size=1024):
        file.write(data)
        bar.update(len(data))

print("Download finished!")

fifa_11_plus_manual.pdf: 100%|██████████| 2.69M/2.69M [00:01<00:00, 2.04MB/s]

Download finished!


In [ ]:
import fitz
import pandas as pd
import re


def extract_fifa11plus_protocol(pdf_path):

    print(f"Starting FIFA 11+ extraction from: {pdf_path}")

    doc = fitz.open(pdf_path)

    dataset = []

    # ==============================
    # STATE VARIABLES
    # ==============================
    current_phase = "Unknown"
    base_exercise_name = None
    current_level = None
    current_dosage = "Not Specified"
    text_buffer = []

    # ==============================
    # REGEX PATTERNS
    # ==============================

    # Exercise titles (Anchored with $ to prevent catching numbered bullet points)
    base_title_pattern = re.compile(
        r'^\s*([1-9]|1[0-5])[\.\s]+([A-Za-z\s\-\&]+)$'
    )

    # Level patterns
    level_pattern = re.compile(
        r'^(Level\s*[1-3])(?:\s*[:\-]?\s*(.*))?',
        re.IGNORECASE
    )

    # Dosage patterns
    dosage_pattern = re.compile(
        r'(\d+\s*x\s*\d+\s*(?:sec|seconds|m|meters|reps)|'
        r'\d+(?:-\d+)?\s*(?:sets|reps|repetitions|sec|seconds|m|meters))',
        re.IGNORECASE
    )

    # Remove noise
    noise_pattern = re.compile(
        r'(fifa\.com|f-marc|copyright|all rights reserved)',
        re.IGNORECASE
    )

    # ==============================
    # SAVE CURRENT EXERCISE
    # ==============================

    def save_current():

        if base_exercise_name and text_buffer:

            name = base_exercise_name

            if current_level:
                name = f"{base_exercise_name} - {current_level}"

            dataset.append(
                {
                    "exercise_name": name.strip(),
                    "phase": current_phase,
                    "duration_or_reps": current_dosage,
                    "instructions": " ".join(text_buffer).strip(),
                    "source": "FIFA 11+ Manual"
                }
            )

    # ==============================
    # TEXT CLEANING
    # ==============================

    def clean_text(text):

        text = re.sub(r'\s+', ' ', text)
        text = text.strip()

        return text

    # ==============================
    # PHASE DETECTION
    # ==============================

    def detect_phase(text):

        nonlocal current_phase

        text_lower = text.lower()

        if "part 1" in text_lower or "running exercises" in text_lower:
            current_phase = "Raise / Activate (Part 1)"

        elif "part 2" in text_lower or "strength" in text_lower:
            current_phase = "Activate / Mobilize (Part 2)"

        elif "part 3" in text_lower or "high-speed" in text_lower:
            current_phase = "Potentiate (Part 3)"

    # ==============================
    # PARSE DOCUMENT
    # ==============================

    for page_num in range(6, doc.page_count):

        page = doc.load_page(page_num)

        # Extract blocks with layout ordering
        blocks = page.get_text("blocks")

        # Sort blocks top-to-bottom
        blocks = sorted(blocks, key=lambda b: (b[1], b[0]))

        for block in blocks:

            text = clean_text(block[4])

            if len(text) < 5:
                continue

            if text.isdigit():
                continue

            if noise_pattern.search(text):
                continue

            # Detect phase
            detect_phase(text)

            # ==========================
            # Detect Exercise Title
            # ==========================

            base_match = base_title_pattern.match(text)

            if base_match:

                save_current()

                base_exercise_name = base_match.group(2).strip()

                current_level = None
                current_dosage = "Not Specified"
                text_buffer = []

                continue

            # ==========================
            # Detect Level
            # ==========================

            level_match = level_pattern.match(text)

            if level_match and base_exercise_name:

                save_current()

                level_num = level_match.group(1)

                level_desc = level_match.group(2) if level_match.group(2) else ""

                current_level = f"{level_num}: {level_desc}".strip(" :")

                current_dosage = "Not Specified"
                text_buffer = []

                continue

            # ==========================
            # Detect Dosage
            # ==========================

            dosage_match = dosage_pattern.search(text)

            if dosage_match:

                current_dosage = dosage_match.group(1)

            # ==========================
            # Capture Instruction Text
            # ==========================

            if base_exercise_name:

                if len(text) > 20:

                    text_buffer.append(text)

    # Save final exercise
    save_current()

    doc.close()

    df = pd.DataFrame(dataset)

    print("Extraction complete!")
    print(f"Exercises extracted: {len(df)}")

    return df


In [ ]:
# 1. Run your exact execution block first
pdf_file = "/content/fifa_11_plus_manual.pdf"
df_fifa = extract_fifa11plus_protocol(pdf_file)

print("Schema perfectly aligned with Gym Dataset!")
print(df_fifa.head())

Starting FIFA 11+ extraction from: /content/fifa_11_plus_manual.pdf
Extraction complete!
Exercises extracted: 12
Schema perfectly aligned with Gym Dataset!
                           exercise_name                      phase  \
0                 RUNNING STRAIGHT AHEAD  Raise / Activate (Part 1)   
1                        RUNNING HIP OUT  Raise / Activate (Part 1)   
2                         RUNNING HIP IN  Raise / Activate (Part 1)   
3               RUNNING CIRCLING PARTNER  Raise / Activate (Part 1)   
4  RUNNING JUMPING WITH SHOULDER CONTACT  Raise / Activate (Part 1)   

  duration_or_reps                                       instructions  \
0              1 M  Important when performing the exercise: Jog st...   
1              1 M  Important when performing the exercise: Jog to...   
2              1 M  Important when performing the exercise: Jog to...   
3    Not Specified  Important when performing the exercise: Jog fo...   
4    Not Specified  Important when performing the ex

In [ ]:
df_fifa.columns

Index(['exercise_name', 'phase', 'duration_or_reps', 'instructions', 'source'], dtype='object')

In [ ]:
df_fifa.tail()

,exercise_name,phase,duration_or_reps,instructions,source
7,Jumping,Raise / Activate (Part 1),2 sets,Important when performing the exercise: This e...,FIFA 11+ Manual
8,RUNNING ACROSS THE PITCH,Raise / Activate (Part 1),1 M,Important when performing the exercise: Run ap...,FIFA 11+ Manual
9,RUNNING BOUNDING,Raise / Activate (Part 1),Not Specified,Important when performing the exercise: Take a...,FIFA 11+ Manual
10,RUNNING PLANT AND CUT,Raise / Activate (Part 1),1 M,Important when performing the exercise: Jog fo...,FIFA 11+ Manual
11,RUNNING PLANT AND CUT,Raise / Activate (Part 1),20 m,APPENDIX: FREQUENTLY ASKED QUESTIONS ABOUT THE...,FIFA 11+ Manual


In [ ]:
df_fifa["instructions"].iloc[5]

'Important when performing the exercise: Run quickly to the second cone then run backwards quickly to the first cone, keeping your hips and knees slightly bent. Repeat, running two cones forwards and one cone back- wards. When you have finished the course, jog back. 1 Make sure you keep your upper body straight. 2 Your hips, knees and feet should be aligned. Do the exercise twice. Do not let your knees buckle inwards.'

In [ ]:
df_fifa.shape

(12, 5)

# generate new needed column

In [ ]:
import os
from groq import Groq
from google.colab import userdata

os.environ["GROQ_API_KEY"] = '######'

client = Groq()
print("Groq client loaded! Ready to use Llama 70B.")

Groq client loaded! Ready to use Llama 70B.


In [ ]:
import json
import pandas as pd

def enrich_warmup_data_llama(row):
    prompt = f"""
    Analyze the following football warm-up exercise extracted from the FIFA 11+ manual.

    Return ONLY a valid JSON object with these exact keys:

    {{
      "primary_muscles": "[String] Name 1-3 primary muscle groups targeted (e.g., 'Hamstrings, Glutes', 'Quadriceps', 'Core'). If it's pure running, use 'Full Lower Body'.",
      "equipment": "[String] Choose from: 'None', 'Cones', 'Partner', 'Ball', 'Partner & Ball'.",
      "primary_action": "[String] Choose exactly ONE: ['general_movement', 'defense', 'dribbling', 'pass', 'score', 'save'].",
      "movement_type": "[String] Choose ONE: ['Aerobic', 'Dynamic Stretch', 'Isometric', 'Plyometric', 'Agility']"
    }}

    ### INPUT EXERCISE DATA:
    Exercise Name: {row['exercise_name']}
    Phase: {row['phase']}
    Instructions: {row['instructions']}
    """

    try:
        # Call Llama 3 70B via Groq
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "You are an expert sports scientist and data engineer. You output strict, valid JSON."
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            model="llama-3.3-70b-versatile",
            response_format={"type": "json_object"},
            temperature=0.1 # Low temperature for highly deterministic, factual output
        )

        # Parse the JSON response
        raw_json = chat_completion.choices[0].message.content
        parsed_data = json.loads(raw_json)

        return pd.Series([
            parsed_data.get('primary_muscles', 'Unknown'),
            parsed_data.get('equipment', 'None'),
            parsed_data.get('primary_action', 'general_movement'),
            parsed_data.get('movement_type', 'Unknown')
        ])

    except Exception as e:
        print(f"Error parsing '{row['exercise_name']}': {e}")
        # Safe fallback so the loop doesn't crash
        return pd.Series(['Unknown', 'Unknown', 'general_movement', 'Unknown'])

print("Llama 70B enrichment function defined!")

Llama 70B enrichment function defined!


In [ ]:
print("Sending data to Llama 3 70B for enrichment...")

# Define the names of our new columns
new_columns = ['primary_muscles', 'equipment', 'primary_action', 'movement_type']

# Apply the function to the DataFrame
df_fifa[new_columns] = df_fifa.apply(enrich_warmup_data_llama, axis=1)

print("Enrichment complete! Here is the final structured data:")
display(df_fifa.head(10))


Sending data to Llama 3 70B for enrichment...
Enrichment complete! Here is the final structured data:


,exercise_name,phase,duration_or_reps,instructions,source,primary_muscles,equipment,primary_action,movement_type
0,RUNNING STRAIGHT AHEAD,Raise / Activate (Part 1),1 M,Important when performing the exercise: Jog st...,FIFA 11+ Manual,Full Lower Body,Cones,general_movement,Aerobic
1,RUNNING HIP OUT,Raise / Activate (Part 1),1 M,Important when performing the exercise: Jog to...,FIFA 11+ Manual,"Glutes, Hip Flexors",Cones,general_movement,Dynamic Stretch
2,RUNNING HIP IN,Raise / Activate (Part 1),1 M,Important when performing the exercise: Jog to...,FIFA 11+ Manual,"Glutes, Hip Flexors",Cones,general_movement,Dynamic Stretch
3,RUNNING CIRCLING PARTNER,Raise / Activate (Part 1),Not Specified,Important when performing the exercise: Jog fo...,FIFA 11+ Manual,Full Lower Body,Cones,general_movement,Aerobic
4,RUNNING JUMPING WITH SHOULDER CONTACT,Raise / Activate (Part 1),Not Specified,Important when performing the exercise: Jog to...,FIFA 11+ Manual,Full Lower Body,Cones,general_movement,Plyometric
5,RUNNING QUICK FORWARDS AND BACKWARDS SPRINTS,Raise / Activate (Part 1),1 M,Important when performing the exercise: Run qu...,FIFA 11+ Manual,Full Lower Body,Cones,general_movement,Aerobic
6,RUNNING QUICK FORWARDS AND BACKWARDS SPRINTS,Activate / Mobilize (Part 2),Not Specified,"PART 2: STRENGTH, PLYOMETRICS AND BALANCE EXER...",FIFA 11+ Manual,Full Lower Body,None,general_movement,Plyometric
7,Jumping,Raise / Activate (Part 1),2 sets,Important when performing the exercise: This e...,FIFA 11+ Manual,"Core, Glutes, Hamstrings",None,general_movement,Isometric
8,RUNNING ACROSS THE PITCH,Raise / Activate (Part 1),1 M,Important when performing the exercise: Run ap...,FIFA 11+ Manual,Full Lower Body,None,general_movement,Aerobic
9,RUNNING BOUNDING,Raise / Activate (Part 1),Not Specified,Important when performing the exercise: Take a...,FIFA 11+ Manual,Full Lower Body,None,general_movement,Plyometric


Saved to 'fifa11plus_enriched_rag_ready.csv'


In [ ]:
df_fifa['domain'] = 'Football / Pitch'

In [ ]:
df_fifa.shape

(12, 10)

In [ ]:
df_fifa.head()

,exercise_name,phase,duration_or_reps,instructions,source,primary_muscles,equipment,primary_action,movement_type,domain
0,RUNNING STRAIGHT AHEAD,Raise / Activate (Part 1),1 M,Important when performing the exercise: Jog st...,FIFA 11+ Manual,Full Lower Body,Cones,general_movement,Aerobic,Football / Pitch
1,RUNNING HIP OUT,Raise / Activate (Part 1),1 M,Important when performing the exercise: Jog to...,FIFA 11+ Manual,"Glutes, Hip Flexors",Cones,general_movement,Dynamic Stretch,Football / Pitch
2,RUNNING HIP IN,Raise / Activate (Part 1),1 M,Important when performing the exercise: Jog to...,FIFA 11+ Manual,"Glutes, Hip Flexors",Cones,general_movement,Dynamic Stretch,Football / Pitch
3,RUNNING CIRCLING PARTNER,Raise / Activate (Part 1),Not Specified,Important when performing the exercise: Jog fo...,FIFA 11+ Manual,Full Lower Body,Cones,general_movement,Aerobic,Football / Pitch
4,RUNNING JUMPING WITH SHOULDER CONTACT,Raise / Activate (Part 1),Not Specified,Important when performing the exercise: Jog to...,FIFA 11+ Manual,Full Lower Body,Cones,general_movement,Plyometric,Football / Pitch


## save data on colab

In [ ]:
df_fifa.to_csv("fifa11plus_enriched_rag_ready.csv", index=False)
print("Saved to 'fifa11plus_enriched_rag_ready.csv'")

Saved to 'fifa11plus_enriched_rag_ready.csv'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
import os
file_source="/content/fifa11plus_enriched_rag_ready.csv"
dist_source="/content/drive/MyDrive/SPORT_METADATA/Structure data/warm_up/gym_warmup/"

os.makedirs("/content/drive/MyDrive/SPORT_METADATA/Structure data/warm_up/gym_warmup/",exist_ok=True)

shutil.copy2(file_source,dist_source)

'/content/drive/MyDrive/SPORT_METADATA/Structure data/warm_up/gym_warmup/fifa11plus_enriched_rag_ready.csv'